# 00 - run all (master orchestrator)

Runs the pipeline end to end via `jupyter nbconvert --execute`, in order, each step writing its own executed notebook with outputs. Notebook 09 (Colab GPU) is skipped here since it runs off-machine. Each step reads its own checkpoints, so anything already scored doesn't get redone.

Run any step on its own by opening its notebook directly. This one's just the one-click 'reproduce everything' entry point.

In [ ]:
import sys
from pathlib import Path
_root = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p/".projectroot").exists())
sys.path.insert(0, str(_root))
from paths import RAW, INTERIM, PROCESSED, FIGURES, TABLES, PROJECT_ROOT
SRC = _root / "src"


In [ ]:
import subprocess, sys, time
STEPS=['02_load_data_and_splits','03_score_plm_masked_marginal','04_score_plm_embeddings',
       '05_esmfold_structure','06_build_features_and_benchmark','07_significance_and_figures',
       '08_results_document']
nbdir=_root/'notebooks'
for s in STEPS:
    t0=time.time(); print(f'=== {s} ===', flush=True)
    cmd=['jupyter','nbconvert','--to','notebook','--execute','--inplace',
         '--ExecutePreprocessor.timeout=-1', str(nbdir/f'{s}.ipynb')]
    r=subprocess.run(cmd)
    status='PASS' if r.returncode==0 else 'FAIL'
    print(f'--- {s}: {status} ({time.time()-t0:.0f}s) ---', flush=True)
    if r.returncode!=0: raise SystemExit(f'{s} failed')
print('ALL STEPS PASS')